# Faruq-v3 — AF2 + IGEM1 direct pair
Seed-42 validation screening. AF2 is active during train and inference; IGEM1 and its auxiliary loss are unchanged. No test access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
BRANCH='codex/af2-strong-model-pairs'
REPO=Path('/content/coffee-bean-detection'); os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone)
    if result.returncode==0: break
    os.chdir('/content')
    if REPO.exists(): shutil.rmtree(REPO)
    time.sleep(2)
else: raise RuntimeError('Git clone gagal tiga kali.')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.drive_project import resolve_drive_project_root, require_project_artifact
REL_ARCHIVE='bundles/faruq-development-v3-grouped.tar'
REL_D0='experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt'
REL_BASE='experiments/faruq-v3-breadth-screening-batch-v1/candidates/IGEM/IGEM1_seed42/weights/best.pt'
PROJECT=resolve_drive_project_root(required_relative_paths=(REL_ARCHIVE,REL_D0,REL_BASE))
ARCHIVE=require_project_artifact(PROJECT,REL_ARCHIVE); D0=require_project_artifact(PROJECT,REL_D0); BASE=require_project_artifact(PROJECT,REL_BASE)
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert len(list((DATA/'train/labels').glob('*.txt')))==1665
assert len(list((DATA/'val/labels').glob('*.txt')))==294
assert not (DATA/'test').exists()
OUTPUT=PROJECT/'experiments/faruq-v3-af2-strong-pairs-v1/AF2IGEM1'; OUTPUT.mkdir(parents=True,exist_ok=True)
print('GPU:',torch.cuda.get_device_name(0)); print('PROJECT:',PROJECT); print('OUTPUT:',OUTPUT)

In [ ]:
ARM='AF2IGEM1'; LOG=OUTPUT/f'{ARM}_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_strong_pair','--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--d0-checkpoint',str(D0),'--standalone-checkpoint',str(BASE),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
print('START/RESUME:',ARM,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream:
    process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
    shown=-1
    while process.poll() is None:
        csv=OUTPUT/f'{ARM}_seed42/results.csv'
        epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        bucket=epochs//5
        if bucket!=shown: print(f'{ARM}: {epochs}/50 epoch tercatat',flush=True); shown=bucket
        time.sleep(30)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
print('SELESAI:',ARM)

In [ ]:
import pandas as pd
from IPython.display import display
SUMMARY=OUTPUT/'val_reports/AF2IGEM1_seed42_decision.json'
result=json.loads(SUMMARY.read_text(encoding='utf-8'))
table=pd.DataFrame(result['values']).T[list(result['delta_pair_minus_standalone'])]
display(table.style.format('{:.2%}'))
print('DELTA:',result['delta_pair_minus_standalone']); print('DECISION:',result['decision']); print('TEST:',result['test_opened']); print('SUMMARY:',SUMMARY)